In [2]:
import pandas as pd 
import numpy as np 
import lightgbm as lgb 


In [3]:
train = pd.read_csv("/Users/mugundan/Documents/dsp/calamity-main/raw_data/train.csv")

In [4]:
train.head()

,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56


In [ ]:
train['road_type'].unique()

array(['urban', 'rural', 'highway'], dtype=object)

In [7]:
train['num_lanes'].unique()

array([2, 4, 1, 3])

In [8]:
train['curvature'].max()

1.0

In [10]:
train['curvature'].min()

0.0

In [11]:
train['speed_limit'].min()

25

In [12]:
train['speed_limit'].max()

70

In [15]:
from sklearn.preprocessing import OneHotEncoder


encoder = OneHotEncoder(sparse_output=False)

encoded = encoder.fit_transform(train[["road_type"]])

# Put back into DataFrame
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(["road_type"]))

# Merge with original df (drop road_type if you don’t need it)
train = train.join(encoded_df)

print(train)

            id road_type  num_lanes  curvature  speed_limit  lighting weather  \
0            0     urban          2       0.06           35  daylight   rainy   
1            1     urban          4       0.99           35  daylight   clear   
2            2     rural          4       0.63           70       dim   clear   
3            3   highway          4       0.07           35       dim   rainy   
4            4     rural          1       0.58           60  daylight   foggy   
...        ...       ...        ...        ...          ...       ...     ...   
517749  517749   highway          4       0.10           70  daylight   foggy   
517750  517750     rural          4       0.47           35  daylight   rainy   
517751  517751     urban          4       0.62           25  daylight   foggy   
517752  517752   highway          3       0.63           25     night   clear   
517753  517753   highway          2       0.31           45       dim   rainy   

        road_signs_present 

In [18]:
from sklearn.preprocessing import OneHotEncoder


encoder = OneHotEncoder(sparse_output=False)

encoded = encoder.fit_transform(train[["lighting"]])

# Put back into DataFrame
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(["lighting"]))

# Merge with original df (drop road_type if you don’t need it)
train = train.join(encoded_df)

print(train)

            id road_type  num_lanes  curvature  speed_limit  lighting weather  \
0            0     urban          2       0.06           35  daylight   rainy   
1            1     urban          4       0.99           35  daylight   clear   
2            2     rural          4       0.63           70       dim   clear   
3            3   highway          4       0.07           35       dim   rainy   
4            4     rural          1       0.58           60  daylight   foggy   
...        ...       ...        ...        ...          ...       ...     ...   
517749  517749   highway          4       0.10           70  daylight   foggy   
517750  517750     rural          4       0.47           35  daylight   rainy   
517751  517751     urban          4       0.62           25  daylight   foggy   
517752  517752   highway          3       0.63           25     night   clear   
517753  517753   highway          2       0.31           45       dim   rainy   

        road_signs_present 

In [19]:
from sklearn.preprocessing import OneHotEncoder


encoder = OneHotEncoder(sparse_output=False)

encoded = encoder.fit_transform(train[["weather"]])

# Put back into DataFrame
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(["weather"]))

# Merge with original df (drop weather if you don’t need it)
train = train.join(encoded_df)

print(train)

            id road_type  num_lanes  curvature  speed_limit  lighting weather  \
0            0     urban          2       0.06           35  daylight   rainy   
1            1     urban          4       0.99           35  daylight   clear   
2            2     rural          4       0.63           70       dim   clear   
3            3   highway          4       0.07           35       dim   rainy   
4            4     rural          1       0.58           60  daylight   foggy   
...        ...       ...        ...        ...          ...       ...     ...   
517749  517749   highway          4       0.10           70  daylight   foggy   
517750  517750     rural          4       0.47           35  daylight   rainy   
517751  517751     urban          4       0.62           25  daylight   foggy   
517752  517752   highway          3       0.63           25     night   clear   
517753  517753   highway          2       0.31           45       dim   rainy   

        road_signs_present 

In [20]:
train.head()

,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,...,accident_risk,road_type_highway,road_type_rural,road_type_urban,lighting_daylight,lighting_dim,lighting_night,weather_clear,weather_foggy,weather_rainy
0,0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,...,0.13,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
1,1,urban,4,0.99,35,daylight,clear,True,False,evening,...,0.35,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
2,2,rural,4,0.63,70,dim,clear,False,True,morning,...,0.30,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
3,3,highway,4,0.07,35,dim,rainy,True,True,morning,...,0.21,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,4,rural,1,0.58,60,daylight,foggy,False,False,evening,...,0.56,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [23]:
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import pickle 


# Suppose train is your DataFrame and "accident_risk" is the target column
X = train[['road_type_highway', 'road_type_rural', 'road_type_urban', 'lighting_daylight', 'lighting_dim', 'lighting_night', 'weather_clear', 'weather_foggy', 'weather_rainy', 'speed_limit', 'curvature']]
y = train["accident_risk"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Initialize LightGBM Regressor
model = lgb.LGBMRegressor(
    objective="regression",
    boosting_type="gbdt",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31
)

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
mse = mean_squared_error(y_test, y_pred)
print("MSE:", mse)

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

# ✅ Load model back
with open("lightgbm_regressor.pkl", "rb") as f:
    loaded_model = pickle.load(f)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002344 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 155
[LightGBM] [Info] Number of data points in the train set: 414203, number of used features: 11
[LightGBM] [Info] Start training from score 0.352605
MSE: 0.0037891898187895298
